<a href="https://colab.research.google.com/github/siddharthshuklacs24/sem-5-AI/blob/main/1BF24CS289_Week9_FOL_Resolution_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
Resolution refutation for First-Order Logic (clausal form)
Conventions:
  - Variables are lowercase (x, y); constants are capitalized (John, Peanuts)
  - Atoms are tuples: ("Likes", "John", "x")
  - A literal is (negated, atom): (True, atom) means ¬atom
  - A clause is a frozenset of literals (a disjunction)
  - The empty frozenset() is the contradiction □
"""

# ---------- Unification (from Week 7) ----------

def is_variable(term):
    return isinstance(term, str) and term[0].islower()

def is_compound(term):
    return isinstance(term, tuple)

def substitute(term, theta):
    if is_variable(term):
        return substitute(theta[term], theta) if term in theta else term
    if is_compound(term):
        return (term[0],) + tuple(substitute(a, theta) for a in term[1:])
    return term

def occurs_in(var, term, theta):
    if var == term:
        return True
    if is_variable(term) and term in theta:
        return occurs_in(var, theta[term], theta)
    if is_compound(term):
        return any(occurs_in(var, a, theta) for a in term[1:])
    return False

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    if occurs_in(var, x, theta):
        return None
    new_theta = dict(theta)
    new_theta[var] = x
    return new_theta

def unify(x, y, theta):
    if theta is None:
        return None
    if x == y:
        return theta
    if is_variable(x):
        return unify_var(x, y, theta)
    if is_variable(y):
        return unify_var(y, x, theta)
    if is_compound(x) and is_compound(y):
        if x[0] != y[0] or len(x) != len(y):
            return None
        for a, b in zip(x[1:], y[1:]):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None

# ---------- Resolution ----------

def rename_term(term, f):
    """Apply f to every variable name in term."""
    if is_variable(term):
        return f(term)
    if is_compound(term):
        return (term[0],) + tuple(rename_term(a, f) for a in term[1:])
    return term

def canonicalize(clause):
    """Rename variables to v0, v1, ... so the number of distinct clauses stays finite."""
    mapping = {}
    def f(v):
        if v not in mapping:
            mapping[v] = f"v{len(mapping)}"
        return mapping[v]
    return frozenset((neg, rename_term(atom, f)) for neg, atom in sorted(clause, key=str))

def resolve(c1, c2):
    """Return all resolvents of clauses c1 and c2."""
    # Standardize apart: rename c2's variables so they do not clash with c1's
    c2r = frozenset((neg, rename_term(atom, lambda v: v + "'")) for neg, atom in c2)
    results = []
    for lit1 in c1:
        for lit2 in c2r:
            (n1, a1), (n2, a2) = lit1, lit2
            if n1 != n2 and a1[0] == a2[0]:          # opposite signs, same predicate
                theta = unify(a1, a2, {})
                if theta is not None:
                    rest = (c1 - {lit1}) | (c2r - {lit2})
                    new = frozenset((n, substitute(a, theta)) for n, a in rest)
                    results.append(canonicalize(new))
    return results

def to_str(term):
    if is_compound(term):
        return f"{term[0]}({', '.join(to_str(a) for a in term[1:])})"
    return str(term)

def clause_str(clause):
    if not clause:
        return "□"
    return " v ".join(("~" if neg else "") + to_str(atom)
                      for neg, atom in sorted(clause, key=str))

def refute(named, goal_label, max_steps=5000):
    """Set-of-support resolution: try to derive the empty clause."""
    labels = {}
    known = []
    for name, clause in named.items():
        labels[clause] = name
        known.append(clause)
    seen = set(known)
    sos = [named[goal_label]]          # set of support starts with the negated goal
    count = 0
    while sos and count < max_steps:
        c = sos.pop(0)
        for d in list(known):
            for r in resolve(c, d):
                if r in seen:
                    continue
                count += 1
                label = f"R{count}"
                labels[r] = label
                seen.add(r)
                known.append(r)
                sos.append(r)
                print(f"{label}: {clause_str(r)}    (from {labels[c]} and {labels[d]})")
                if not r:
                    return True
    return False

if __name__ == "__main__":
    kb = {
        "a":  {(True, ("Food", "x")), (False, ("Likes", "John", "x"))},
        "b1": {(False, ("Food", "Apple"))},
        "b2": {(False, ("Food", "Vegetables"))},
        "c":  {(True, ("Eats", "x", "y")), (False, ("Killed", "x")), (False, ("Food", "y"))},
        "d1": {(False, ("Eats", "Anil", "Peanuts"))},
        "d2": {(False, ("Alive", "Anil"))},
        "e":  {(True, ("Eats", "Anil", "x")), (False, ("Eats", "Harry", "x"))},
        "f":  {(True, ("Alive", "x")), (True, ("Killed", "x"))},
        "g":  {(False, ("Killed", "x")), (False, ("Alive", "x"))},
        "G":  {(True, ("Likes", "John", "Peanuts"))},   # negated goal ¬Likes(John, Peanuts)
    }

    print("Knowledge base in CNF (G is the negated goal):")
    for name, clause in kb.items():
        print(f"  {name}: {clause_str(frozenset(clause))}")

    named = {name: canonicalize(clause) for name, clause in kb.items()}
    print("\nResolution trace:")
    proved = refute(named, "G")
    print("\nResult:", "PROVED: John likes Peanuts" if proved else "NOT PROVED")

Knowledge base in CNF (G is the negated goal):
  a: Likes(John, x) v ~Food(x)
  b1: Food(Apple)
  b2: Food(Vegetables)
  c: Food(y) v Killed(x) v ~Eats(x, y)
  d1: Eats(Anil, Peanuts)
  d2: Alive(Anil)
  e: Eats(Harry, x) v ~Eats(Anil, x)
  f: ~Alive(x) v ~Killed(x)
  g: Alive(x) v Killed(x)
  G: ~Likes(John, Peanuts)

Resolution trace:
R1: ~Food(Peanuts)    (from G and a)
R2: Killed(v0) v ~Eats(v0, Peanuts)    (from R1 and c)
R3: Killed(Anil)    (from R2 and d1)
R4: Killed(Harry) v ~Eats(Anil, Peanuts)    (from R2 and e)
R5: ~Alive(v0) v ~Eats(v0, Peanuts)    (from R2 and f)
R6: ~Alive(Anil)    (from R3 and f)
R7: Killed(Harry)    (from R4 and d1)
R8: ~Alive(Harry) v ~Eats(Anil, Peanuts)    (from R4 and f)
R9: ~Eats(Anil, Peanuts)    (from R5 and d2)
R10: □    (from R6 and d2)

Result: PROVED: John likes Peanuts
